# Jon2 AI — تدريب النموذج الأصلي

هذه الدفترية تدرب محرك Jon2 المكتوب داخل المشروع، ولا تستخدم Qwen أو GPT أو أي API خارجي. التدريب لا يبدأ إلا بعد مراجعة البيانات وتوسيعها؛ العينة الحالية ما زالت صغيرة ولا تصنع مساعدًا عامًا قويًا.

**سياسة البيانات:** استخدم نصوصًا أصلية أو مرخصة صراحةً للتدريب والاستخدام المقصود. لا تستخدم كتبًا أو مواقع أو محادثات خاصة دون إذن موثق.

In [ ]:
# استنساخ المشروع
!git clone https://github.com/jonkabeel-star/jon2-ai.git
%cd /content/jon2-ai/ai_engine


In [ ]:
# فحص حجم البيانات ومصادرها قبل التدريب
from pathlib import Path
files = sorted(Path('training_data').rglob('*.txt'))
for file in files:
    lines = [line for line in file.read_text(encoding='utf-8').splitlines() if not line.lstrip().startswith('#')]
    print(f'{file}: {len(chr(10).join(lines)):,} حرف')
total = sum(len('\n'.join([line for line in f.read_text(encoding='utf-8').splitlines() if not line.lstrip().startswith('#')])) for f in files)
print(f'إجمالي النصوص: {total:,} حرف')
print('الحد الإرشادي المقترح قبل تجربة التدريب:', f'{100_000:,} حرف على الأقل')
if total < 100_000:
    print('توقف: البيانات الحالية قليلة. لا نكرر النصوص ولا نضيف نصوصًا غير مرخصة لرفع العدد.')
else:
    print('الحجم تجاوز الحد الإرشادي الأولي؛ راجع جودة كل مصدر وتسجيل الحقوق قبل التدريب.')


## مراجعة الحقوق والجودة

لا تتابع التدريب لمجرد تجاوز عدد الحروف. يجب أن تكون الأمثلة صحيحة ومتنوعة وغير مكررة، وأن تكون حقوق كل ملف مسجلة في `DATASET_REGISTER.md`. حد 100 ألف حرف مجرد بوابة أولية للتجربة، وليس حجمًا كافيًا بالضرورة لمساعد عام.

In [ ]:
# شغّل هذه الخلية فقط بعد اكتمال مراجعة البيانات
from pathlib import Path
import subprocess, sys
from jon2_core import read_corpus
corpus = read_corpus(Path('training_data'))
if len(corpus) < 100_000:
    raise RuntimeError('التدريب متوقف: وسّع البيانات الأصلية أو المرخصة وراجع جودتها أولًا.')
subprocess.run([
    sys.executable, 'jon2_core.py',
    '--data', 'training_data',
    '--output', 'checkpoints/jon2-small.pt',
    '--steps', '5000',
    '--batch-size', '16',
    '--context', '128',
], check=True)


In [ ]:
# اختبار تحميل أوزان Jon2 التي تم تدريبها
from pathlib import Path
from jon2_core import load_model, generate
checkpoint = Path('checkpoints/jon2-small.pt')
assert checkpoint.is_file(), 'ملف الأوزان غير موجود؛ لم يكتمل التدريب.'
model, tokenizer, device = load_model(str(checkpoint))
prompt = 'المستخدم: مرحبًا، من أنت؟\\nJon2:'
result = generate(model, tokenizer, prompt, device, max_new_chars=240, temperature=0.7)
print(result)
print('مهم: راجع الإجابة يدويًا؛ نجاح التحميل لا يثبت جودة الذكاء.')


## ما بعد الاختبار

لا تنشر النموذج قبل تقييمه على مجموعة أسئلة منفصلة لم تدخل التدريب، وفحص العربية والهلوسة والتكرار والسلامة. لا ترفع ملف الأوزان إلى مستودع GitHub العام؛ خزّنه في مساحة خاصة آمنة عند تجهيز الاستضافة.